# 02 – Data Storage Layer

**Projekt:** Wohnungsmarkt-Radar Wien – Preis/Leistungs-Score für Mietwohnungen

**Layer:** Data Storage Layer - Daten in geeigneter Form speichern (Rohzone und bereinigte Zone in MongoDB)

**Infrastruktur:**
MongoDB läuft als Docker-Container wohnung_mongodb (mongo:4.4), gestartet per docker compose up -d. Kafka und Zookeeper (Nachrichten, Source/Storage) und Spark (Analysis Layer) laufen im selben Compose.

**Beschreibung:** Dieses Notebook beschreibt, wie die gescrapten Daten in MongoDB gespeichert werden. Es holt die Daten aus dem raw/runs Ordner und speichert sie in Collections. Das ganze ist in zwei Zonen aufgeteilt:

| Zone | Collections | Inhalt |
|---|---|---|
| **Rohzone** | `raw_willhaben`, `raw_immoscout24`, | Die Inserate genau so, wie die Scraper sie liefern (unterschiedliche Felder je Quelle), ergänzt um `first_seen`, `last_seen`, `status` |
| **Bereinigte Zone** | `listings_clean` | Einheitliches Schema über alle Quellen, bereinigt (fehlende Werte, Duplikate, ect), mit Lage- und Öffi-Merkmalen |


**Warum erst roh speichern und dann bereinigen?**
- Die Rohdaten bleiben erhalten. Ändern wir später eine Bereinigungsregel, müssen wir nicht nochmal scrapen.
- Die Quellen haben unterschiedliche Attribute. Ein Dokumentenspeicher nimmt das ohne festes Schema auf, das einheitliche Schema entsteht erst in der bereinigten Zone.
- `listings_clean` kann jederzeit neu aufgebaut werden.

**Zustand je Inserat:**
- `first_seen` / `last_seen`: erster und letzter Zeitpunkt, zu dem das Inserat bei einem Lauf gesehen wurde
- `status`: `active` oder `removed`. Ein Inserat gilt als entfernt, wenn es in einem **vollständigen** Lauf nicht mehr auftaucht. Schnelle Läufe (nur erste Seite) reichen dafür nicht, weil ein Inserat nur nach hinten gerutscht sein kann.
- Entfernte Inserate werden nicht gelöscht, sondern markiert. Für den Score werden nur aktive verwendet, die übrigen bleiben für weitere Auswertungen (zB. Dauer online, Verlauf der Preise) erhalten.

## 1. Imports, Konfiguration und Verbindung zu MongoDB

In [1]:
import json
import re
import time
import warnings
from datetime import datetime, timezone
import urllib.parse
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from pyproj import Transformer
from pymongo import MongoClient, UpdateOne
from pymongo.errors import ServerSelectionTimeoutError

# -------------------------------------------------------------------------
# 1. Pfade & externe Dateien
# -------------------------------------------------------------------------
BASE = Path.cwd().parent
RAW_DIR = BASE / "data" / "raw"
STOPS_FILE = BASE / "data" / "external" / "haltestellen.json"
BEZIRKE_FILE = BASE / "data" / "external" / "bezirksgrenzen.json"
BEZIRKE_CRS = "EPSG:31256"    # Koordinatensystem der Bezirksgrenzen (Wien)
OEFFI_RADIUS_M = 500          # Umkreis in Metern für die Anzahl der Linien

# -------------------------------------------------------------------------
# 2. Plausibilitätsgrenzen & Filter-Konstanten
# -------------------------------------------------------------------------
PRICE_RANGE = (200, 20_000)   # Miete pro Monat in Euro
AREA_RANGE = (10, 400)        # Wohnfläche in m²
VIENNA_LAT = (48.10, 48.35)   # Bounding-Box Wien Breite
VIENNA_LON = (16.17, 16.60)   # Bounding-Box Wien Länge
MIN_LISTINGS_PER_BEZIRK = 5   # Darunter wird kein Bezirks-Median berechnet

NOT_APARTMENT_RE = re.compile(r"büro|geschäftslokal|gewerbe|lager|garage|stellplatz|parkplatz", re.I)
EG_RE = re.compile(r"(?i:erdgescho(?:ss|ß)|parterre)|\bEG\b")
DG_RE = re.compile(r"(?i:dachgescho(?:ss|ß)|dachwohnung)|\bDG\b")

# Erweiterte Linientypen
LTYP_KIND = {
    1: "tram",
    2: "bus",
    3: "regionalbus",
    4: "ubahn",
    5: "sbahn",
    6: "badnerbahn"
}

# -------------------------------------------------------------------------
# 3. Verbindung zu MongoDB & Collections
# -------------------------------------------------------------------------
MONGO_URI = "mongodb://localhost:27017/"
DB_NAME = "wohnungsmarkt"  

client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)

try:
    client.server_info()
    db = client[DB_NAME]
    
    print(f"Erfolgreich mit MongoDB verbunden (Datenbank: '{DB_NAME}')")
except ServerSelectionTimeoutError as e:
    print(f"Fehler: Konnte keine Verbindung zu MongoDB herstellen ({e})")

Erfolgreich mit MongoDB verbunden (Datenbank: 'wohnungsmarkt')


## 2. Rohzone - Läufe in MongoDB laden

Jede Zeile aus den runs wird als Dokument gespeichert mit dem Key/Value quelle:id als _id

- neues Inserat -> wird eingefügt, first_seen = Zeitpunkt des Laufs
- bekanntes Inserat -> Felder werden aktualisiert, last_seen wird aktualisiert
- nach einem Full-run -> aktive Inserate die nicht mehr vorkommen werden als removed markiert
- Die Läufe werden in zeitlicher Reihenfolge verabeitet. Bereits geladene Dateien merkt sich die Collection _loaded_runs -> neues Ausführen lädt also nichts doppelt

In [2]:
def read_jsonl(path):
    rows, bad = [], 0
    if not path.exists():
        print(f"{path.name} nicht gefunden – übersprungen")
        return rows
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                bad += 1
    print(f"   {path.name}: {len(rows)} Zeilen" + (f" ({bad} defekt)" if bad else ""))
    return rows


SOURCES = {
    "willhaben":   {"collection": "raw_willhaben",   "id_field": "id"},
    "immoscout24": {"collection": "raw_immoscout24", "id_field": "exposeId"},
}
FULL_RUN_MIN_SHARE = 0.5     # ein voller Lauf, der weniger als die Hälfte der bisher aktiven Inserate liefert, markiert nichts als entfernt


def sanitize(value):
    """MongoDB-Feldnamen dürfen keinen Punkt enthalten und nicht mit '$' beginnen."""
    if isinstance(value, dict):
        return {k.replace(".", "_").lstrip("$"): sanitize(v) for k, v in value.items()}
    if isinstance(value, list):
        return [sanitize(v) for v in value]
    return value


def parse_dt(text):
    return datetime.fromisoformat(text.replace("Z", "+00:00")) if text else None


def upsert_docs(source, run_type, docs):
    """Schreibt die Inserate eines Laufs in die Rohzone (Upsert) und markiert nach einem vollen Lauf entfernte Inserate."""
    cfg = SOURCES[source]
    coll = db[cfg["collection"]]
    docs = [d for d in docs if d.get(cfg["id_field"]) is not None]
    if not docs:
        return 0
    run_time = max(parse_dt(d.get("_scraped_at")) for d in docs)
    active_before = coll.count_documents({"status": "active"})

    ops = []
    for d in docs:
        seen = parse_dt(d.get("_scraped_at"))
        fields = sanitize({k: v for k, v in d.items() if k not in ("_scraped_at", "_run_type")})
        ops.append(UpdateOne(
            {"_id": f"{source}:{d[cfg['id_field']]}"},
            {"$set": {**fields, "status": "active", "last_run_type": run_type},
             "$max": {"last_seen": seen},
             "$setOnInsert": {"first_seen": seen}},
            upsert=True))
    for i in range(0, len(ops), 1000):
        coll.bulk_write(ops[i:i + 1000], ordered=False)

    # Voller Lauf: was nicht mehr auftaucht, ist entfernt (nur wenn der Lauf plausibel vollständig ist)
    if run_type == "full":
        if active_before == 0 or len(docs) >= FULL_RUN_MIN_SHARE * active_before:
            res = coll.update_many({"status": "active", "last_seen": {"$lt": run_time}},
                                   {"$set": {"status": "removed", "removed_at": run_time}})
            if res.modified_count:
                print(f"   {source}: {res.modified_count} Inserate als entfernt markiert")
        else:
            print(f"{source}: voller Lauf liefert nur {len(docs)} von {active_before} aktiven Inseraten – "
                  "nichts als entfernt markiert (Lauf unvollständig?)")
    return len(docs)

def load_run_file(path, source, run_type):
    return upsert_docs(source, run_type, read_jsonl(path))

def load_all_runs():
    loaded = {d["_id"] for d in db["_loaded_runs"].find({}, {"_id": 1})}
    files = sorted((RAW_DIR / "runs").glob("*.jsonl"), key=lambda f: f.stem.split("_", 2)[2])
    new_files = [f for f in files if f.name not in loaded]
    print(f"{len(files)} Lauf-Dateien, davon {len(new_files)} noch nicht geladen")
    for f in new_files:
        source, run_type, _ = f.stem.split("_", 2)
        if source not in SOURCES:
            continue
        n = load_run_file(f, source, run_type)
        db["_loaded_runs"].insert_one({"_id": f.name, "loaded_at": datetime.now(timezone.utc), "docs": n})
        print(f"   {f.name}: {n} Inserate")
    for cfg in SOURCES.values():
        db[cfg["collection"]].create_index([("status", 1), ("last_seen", 1)])


load_all_runs()

6 Lauf-Dateien, davon 6 noch nicht geladen
   immoscout24_fast_2026-11-28T09-27-00.jsonl: 20 Zeilen
   immoscout24_fast_2026-11-28T09-27-00.jsonl: 20 Inserate
   willhaben_fast_2026-11-28T09-27-00.jsonl: 25 Zeilen
   willhaben_fast_2026-11-28T09-27-00.jsonl: 25 Inserate
   immoscout24_fast_2026-11-28T09-27-29.jsonl: 20 Zeilen
   immoscout24_fast_2026-11-28T09-27-29.jsonl: 20 Inserate
   willhaben_fast_2026-11-28T09-27-29.jsonl: 25 Zeilen
   willhaben_fast_2026-11-28T09-27-29.jsonl: 25 Inserate
   immoscout24_fast_2026-11-28T09-51-57.jsonl: 20 Zeilen
   immoscout24_fast_2026-11-28T09-51-57.jsonl: 20 Inserate
   willhaben_fast_2026-11-28T09-51-57.jsonl: 25 Zeilen
   willhaben_fast_2026-11-28T09-51-57.jsonl: 25 Inserate


**Consumer:**

In [3]:
import time
from collections import defaultdict
from confluent_kafka import Consumer

KAFKA_BROKER = "localhost:9092"      
KAFKA_TOPIC = "listings.raw"


def consume_runs(group_id="storage-loader", start_wait_s=15, idle_s=5):
    """Liest neue Nachrichten aus Kafka und schreibt sie in die Rohzone."""
    consumer = Consumer({"bootstrap.servers": KAFKA_BROKER, "group.id": group_id,
                         "auto.offset.reset": "earliest", "enable.auto.commit": False})
    consumer.subscribe([KAFKA_TOPIC])

    messages = []
    deadline = time.time() + start_wait_s          # beim ersten Start dauert der Verbindungsaufbau etwas
    while time.time() < deadline:
        msg = consumer.poll(1.0)
        if msg is None:
            continue
        if msg.error():
            print("Kafka:", msg.error())
            continue
        messages.append(msg)
        deadline = time.time() + idle_s            # Ende, wenn 5 Sekunden nichts Neues kommt
    print(f"{len(messages)} Nachrichten gelesen")

    # Nachrichten zu Läufen zusammensetzen: ein Lauf = gleiche Quelle, Laufart und _scraped_at
    runs = defaultdict(list)
    for m in messages:
        doc = json.loads(m.value().decode("utf-8"))
        source = m.key().decode("utf-8").split(":", 1)[0]
        runs[(source, doc.get("_run_type", "fast"), doc["_scraped_at"])].append(doc)

    for source, run_type, scraped_at in sorted(runs, key=lambda k: k[2]):
        if source not in SOURCES:                 
            continue
        n = upsert_docs(source, run_type, runs[(source, run_type, scraped_at)])
        print(f"   {source} ({run_type}, {scraped_at[:19]}): {n} Inserate")

    if messages:
        consumer.commit(asynchronous=False)        # erst nach dem Schreiben bestätigen
    consumer.close()


consume_runs()

135 Nachrichten gelesen
   willhaben (fast, 2026-11-28T08:27:00): 25 Inserate
   immoscout24 (fast, 2026-11-28T08:27:04): 20 Inserate
   willhaben (fast, 2026-11-28T08:27:29): 25 Inserate
   immoscout24 (fast, 2026-11-28T08:27:33): 20 Inserate
   willhaben (fast, 2026-11-28T08:51:57): 25 Inserate
   immoscout24 (fast, 2026-11-28T08:52:00): 20 Inserate


**Überblick Rohzone:**

In [4]:
overview = pd.DataFrame({
    src: {"Dokumente": db[cfg["collection"]].count_documents({}),
          "aktiv": db[cfg["collection"]].count_documents({"status": "active"}),
          "entfernt": db[cfg["collection"]].count_documents({"status": "removed"})}
    for src, cfg in SOURCES.items()
})
display(overview)

example = db["raw_willhaben"].find_one({}, {"heading": 1, "price": 1,"coordinates": 1, "postcode": 1, "first_seen": 1, "last_seen": 1, "status": 1})
print("Beispieldokument (Willhaben):")
print(example)

,willhaben,immoscout24
Dokumente,30,20
aktiv,30,20
entfernt,0,0


Beispieldokument (Willhaben):
{'_id': 'willhaben:2043700366', 'coordinates': '48.144081115722656,16.291519165039062', 'first_seen': datetime.datetime(2026, 11, 28, 8, 27, 0, 70000), 'heading': '2-Zimmer Mietwohnung in Perfektastraße, beste Infrastruktur', 'last_seen': datetime.datetime(2026, 11, 28, 8, 51, 57, 434000), 'postcode': 1230, 'price': 970, 'status': 'active'}


# 3. Bereinigte Zone - etablieren eines einheitlichen Schemas

Hier definieren wir die Funktionen sowie Beispiele zur Kontrolle und Visualisierung. Aus den zwei Rohzonen entsteht eine Tabelle mit denselben Spalten. Die Bereinigung läuft in diesen Schritten:

1. **Flags hinzufügen & Normalisieren:** Feldnamen je Quelle auf das gemeinsame Schema abbilden (Freifläche vorhanden, Privat-Angebot vorhanden, Gewerbliches-Angebot vorhanden)
2. **Filtern:** keine Büros oder Garagen, Preis und Fläche plausibel, Preis/Fläche/Zimmeranzahl vorhanden, keine Duplikate (sowohl innerhalb der Portale als auch zwischen den Portalen), nur Mietobjekte
3. **Merkmale:** Preis pro m², Bezirk, Vergleich mit dem Bezirksmedian, Öffi-Anbindung (Distanz zu den Öffis, Anzahl der Linien im Umkreis von 500 m, U-Bahn kleiner 500m Flag).
4. **Speichern:** Speichern in `listings_clean`.

**Definition Preis:** Bei allen Quellen die monatlichen Gesamtkosten inklusive Betriebskosten.

**Hilfsfunktionen:**

In [5]:
import re

def num(x):
    try:
        return None if x in (None, "") else float(x)
    except (TypeError, ValueError):
        return None


def parse_coords(s):
    """'48.2351828,16.4810643' -> (lat, lon)"""
    try:
        lat, lon = str(s).split(",")
        return float(lat), float(lon)
    except (ValueError, AttributeError):
        return None, None

# Regex um eine 4-stellige PLZ (aus immoScout24) aus dem addressString zu extrahieren
PLZ_RE = re.compile(r'\b(1\d{3})\b')

def parse_plz(addr_string):
    if not addr_string:
        return None
    match = PLZ_RE.search(addr_string)
    return int(match.group(1)) if match else None

def safe_flag(val):
    if val is True or val == 1 or val == 1.0 or str(val).lower() in ["true", "1"]:
        return 1
    return 0

def record(source, id_, url, title, price, area, rooms, plz, lat, lon, has_outdoor, is_rent, is_buy, is_private, is_commercial, address, street=None):
    return dict(
        source=source, 
        id=str(id_), 
        url=url, 
        title=title, 
        plz=int(plz) if plz is not None else None, 
        address=address, 
        street=street,
        price=num(price), 
        area_m2=num(area), 
        rooms=num(rooms), 
        has_outdoor_space=int(has_outdoor),
        is_rent=int(is_rent),
        is_buy=int(is_buy),
        is_private=int(is_private),
        is_commercial=int(is_commercial),
        lat=lat, 
        lon=lon
    )

**1. Normalisieren & Flags setzen:**

In [6]:
def norm_willhaben(r):
    lat, lon = parse_coords(r.get("coordinates"))
    #Hilfsvariablen
    seo = r.get("seo_url")
    seo_url_lower = str(seo).lower() if seo else ""

    raw_price = r.get("price", 0)
    try:
        price_val = float(raw_price) if raw_price is not None else 0
    except (ValueError, TypeError):
        price_val = 0

    #Anbieter-Flags definieren
    is_private = safe_flag(r.get("isprivate"))
    is_commercial = 1 - is_private
    has_rent_key = "rent/per_month_lettings" in r or (0 < price_val < 50000)
    is_rent = 1 if ("/mietwohnungen/" in seo_url_lower or has_rent_key) else 0
    is_buy = 1 - is_rent

    # Outdoor-Flag
    free_area_name = str(r.get("free_area_type_name", "")).lower()
    free_area_total = r.get("free_area/free_area_area_total", 0) or 0
    has_outdoor = 1 if (free_area_name in ["balkon", "terrasse", "garten", "loggia"] or free_area_total > 0) else 0

    return record(
        "willhaben", r.get("id"),
        f"https://www.willhaben.at/iad/{seo}" if seo else None,
        r.get("heading"),
        r.get("price"),
        r.get("estate_size") or r.get("estate_size/living_area"),
        r.get("number_of_rooms"),
        num(r.get("postcode")),
        lat, lon,
        has_outdoor,
        is_rent,
        is_buy,
        is_private,
        is_commercial,
        r.get("location"))


def norm_immoscout(r, stats):
    meta = r.get("meta") or {}
    transfer_type = str(meta.get("transferType", "")).upper()
    
    if meta.get("transferType") != "RENT":
        stats["kein_rent"] += 1
        return None
    if meta.get("hasChildren") or str(r.get("displayType", "")).startswith("DEVELOPER"):
        stats["projekt"] += 1          # Bauprojekte haben Spannen statt Einzelwerten
        return None
    
    loc = r.get("location") or {}
    addr = r.get("addressString") or ""

    # Prüfe, ob es eine Freifläche gibt
    has_outdoor = 0
    badges = r.get("typedBadges", [])
    if isinstance(badges, list):
        for b in badges:
            details = b.get("details", {}) if isinstance(b, dict) else {}
            val = str(details.get("value", "")).upper()
            if any(kw in val for kw in ["BALCONY", "TERRACE", "GARDEN", "LOGGIA"]):
                has_outdoor = 1
                break
    if not has_outdoor and any(kw in str(r).upper() for kw in ["BALCONY", "BALKON", "TERRACE", "TERRASSE"]):
        has_outdoor = 1

    # PLZ aus addressString extrahieren
    plz_val = parse_plz(addr)

    # Zur Sicherheit prüfen, ob es Kauf ist
    is_rent = 1 if transfer_type == "RENT" else 0
    is_buy = 1 if transfer_type != "RENT" else 0

    # Anbieter-Flags definieren
    is_private = safe_flag(r.get("isPrivate"))
    is_commercial = 1 - is_private

    return record(
        "immoscout24", 
        r.get("exposeId"),
        (r.get("links") or {}).get("absoluteURL"),
        r.get("headline"),
        r.get("primaryPrice"), 
        r.get("primaryArea"), 
        r.get("numberOfRooms"),
        plz_val,
        num(loc.get("lat")), 
        num(loc.get("lon")),
        has_outdoor,
        is_rent,
        is_buy,
        is_private,
        is_commercial,
        addr
    )


Beispiel:

In [7]:
import json

# Funktion zum Finden und Ausgeben des ersten validen Eintrags
def print_first_valid(source_name, collection, norm_func):
    for raw in collection.find():
        # Da norm_immoscout ein stats-Dict verlangt:
        if source_name == "immoscout24":
            stats = {"kein_rent": 0, "projekt": 0}
            res = norm_func(raw, stats)
        else:
            res = norm_func(raw)
            
        if res:  # Wenn es nicht durchgefiltert wurde
            print(f"\n--- BEISPIEL: {source_name.upper()} (Normalisiert) ---")
            print(json.dumps(res, indent=2, ensure_ascii=False))
            break

# Ausführen für beide
print_first_valid("willhaben", db["raw_willhaben"], norm_willhaben)
print_first_valid("immoscout24", db["raw_immoscout24"], norm_immoscout)


--- BEISPIEL: WILLHABEN (Normalisiert) ---
{
  "source": "willhaben",
  "id": "2043700366",
  "url": "https://www.willhaben.at/iad/immobilien/d/mietwohnungen/wien/wien-1230-liesing/2-zimmer-mietwohnung-in-perfektastrasse-beste-infrastruktur-2043700366/",
  "title": "2-Zimmer Mietwohnung in Perfektastraße, beste Infrastruktur",
  "plz": 1230,
  "address": "Wien, 23. Bezirk, Liesing",
  "street": null,
  "price": 970.0,
  "area_m2": 50.0,
  "rooms": 2.0,
  "has_outdoor_space": 1,
  "is_rent": 1,
  "is_buy": 0,
  "is_private": 0,
  "is_commercial": 1,
  "lat": 48.144081115722656,
  "lon": 16.291519165039062
}

--- BEISPIEL: IMMOSCOUT24 (Normalisiert) ---
{
  "source": "immoscout24",
  "id": "6b0404cb24a9ec393adb2191",
  "url": "https://www.immobilienscout24.at/expose/6b0404cb24a9ec393adb2191",
  "title": "wunderschöne Dachgeschoßwohnung",
  "plz": 1040,
  "address": "1040 Wien",
  "street": null,
  "price": 2490.0,
  "area_m2": 104.0,
  "rooms": 2.5,
  "has_outdoor_space": 1,
  "is_rent"

**Öffi-Haltestellen einlesen & für jede Wohnung nearest ubahn, Anzahl der jeweiligen Öffis, sowie die Gesamtanzahl der Öffis im Umkreis von 500 Metern :**

In [8]:
# Dein neues Mapping als globale Konstante
LTYP_KIND = {
    1: "tram",
    2: "bus",
    3: "regionalbus",
    4: "ubahn",
    5: "sbahn",
    6: "badnerbahn"
}

def load_stops(path):
    """Wiener Linien OGD-Haltestellen angepasst an LTYP_KIND."""
    with open(path, encoding="utf-8") as f:
        features = json.load(f)["features"]
        
    lat, lon, kind, lines, names = [], [], [], [], []
    for ft in features:
        geom, props = ft.get("geometry") or {}, ft.get("properties") or {}
        if geom.get("type") != "Point":
            continue
            
        # Linien bereinigen (Nachtlinien ignorieren)
        tokens = {t.strip() for t in str(props.get("HLINIEN") or "").split(",") if t.strip()}
        day_lines = frozenset(t for t in tokens if not NIGHT_LINE_RE.match(t))
        if not day_lines:
            continue
            
        # Typ über LTYP_KIND ermitteln
        ltyp = props.get("LTYP")
        k = LTYP_KIND.get(ltyp)
        if not k:
            continue  # Unbekannte oder ignorierte Typen überspringen
            
        lon_, lat_ = geom["coordinates"][:2]  # GeoJSON: erst lon, dann lat!
        lat.append(lat_)
        lon.append(lon_)
        kind.append(k)
        lines.append(day_lines)
        names.append(props.get("HTXT"))
        
    stops = dict(
        lat=np.radians(lat), 
        lon=np.radians(lon), 
        kind=np.array(kind), 
        lines=lines,
        names=np.array(names, dtype=object)
    )
    
    # Counts für alle definierten Typen ausgeben
    valid_kinds = list(LTYP_KIND.values())
    counts = {k: int((stops["kind"] == k).sum()) for k in valid_kinds}
    counts_str = ", ".join([f"{k.capitalize()} {counts[k]}" for k in valid_kinds])
    print(f"   {len(kind)} Haltestellen geladen ({counts_str})")
    
    if counts["ubahn"] == 0:
        print("Keine U-Bahn-Haltestellen erkannt – bitte LTYP / HLINIEN prüfen.")
        
    return stops


def oeffi_features(lat, lon, stops, radius=OEFFI_RADIUS_M, chunk=1000):
    """Je Wohnung: Distanz zur nächsten U-Bahn (m) + Anzahl verschiedener Linien je Typ im Umkreis."""
    lat, lon = np.radians(np.asarray(lat, float)), np.radians(np.asarray(lon, float))
    n = len(lat)
    
    # Output-Dictionary um alle Verkehrsmittel aus LTYP_KIND erweitert
    out = dict(
        dist_ubahn_m=np.full(n, np.nan), 
        nearest_ubahn=np.full(n, None, dtype=object),
        n_ubahn_lines=np.zeros(n, int), 
        n_tram_lines=np.zeros(n, int),
        n_bus_lines=np.zeros(n, int), 
        n_regionalbus_lines=np.zeros(n, int),
        n_sbahn_lines=np.zeros(n, int), 
        n_badnerbahn_lines=np.zeros(n, int),
        n_lines=np.zeros(n, int)
    )
    
    kinds = stops["kind"]
    ubahn_kinds = ["ubahn"]
    u_idx = np.where(np.isin(kinds, ubahn_kinds))[0]
    
    valid_kinds = list(LTYP_KIND.values())
    
    for i in range(0, n, chunk):
        la, lo = lat[i:i + chunk, None], lon[i:i + chunk, None]
        a = np.sin((stops["lat"] - la) / 2) ** 2 + np.cos(la) * np.cos(stops["lat"]) * np.sin((stops["lon"] - lo) / 2) ** 2
        d = 2 * 6_371_000 * np.arcsin(np.sqrt(a))
        
        if len(u_idx):
            du = d[:, u_idx]
            j = du.argmin(axis=1)
            out["dist_ubahn_m"][i:i + chunk] = du[np.arange(len(j)), j]
            out["nearest_ubahn"][i:i + chunk] = stops["names"][u_idx][j]
            
        within = d <= radius
        for r in range(d.shape[0]):
            idx = np.where(within[r])[0]
            by_kind = {k: set() for k in valid_kinds}
            
            for k_idx in idx:
                k_val = kinds[k_idx]
                if k_val in by_kind:
                    by_kind[k_val] |= stops["lines"][k_idx]
                    
            row = i + r
            out["n_ubahn_lines"][row] = len(by_kind["ubahn"])
            out["n_tram_lines"][row] = len(by_kind["tram"])
            out["n_bus_lines"][row] = len(by_kind["bus"])
            out["n_regionalbus_lines"][row] = len(by_kind["regionalbus"])
            out["n_sbahn_lines"][row] = len(by_kind["sbahn"])
            out["n_badnerbahn_lines"][row] = len(by_kind["badnerbahn"])
            
            # Alle Linien über alle Typen hinweg aggregieren (mehrere Haltestellen der gleichen Linie können im Umkreis liegen)
            all_lines = set().union(*by_kind.values())
            out["n_lines"][row] = len(all_lines)
            
    out["dist_ubahn_m"] = out["dist_ubahn_m"].round(0)
    return out

Beispiel:

In [9]:
import json
import numpy as np

def print_first_valid_with_oeffi(source_name, collection, norm_func, stops):
    for raw in collection.find():
        # 1. Normalisierung wie gehabt
        if source_name == "immoscout24":
            stats = {"kein_rent": 0, "projekt": 0}
            res = norm_func(raw, stats)
        else:
            res = norm_func(raw)
            
        if res:  # Wenn das Listing gültig ist (nicht herausgefiltert)
            # 2. Koordinaten aus dem normalisierten Ergebnis holen 
            # (Angenommen res hat keys "lat" und "lon", ansonsten anpassen)
            lat_val = res.get("lat")
            lon_val = res.get("lon")
            
            if lat_val is not None and lon_val is not None:
                # 3. Öffi-Features temporär für diese eine Koordinate berechnen
                oeffi_res = oeffi_features(lat=[lat_val], lon=[lon_val], stops=stops)
                
                # 4. Öffi-Werte sauber als Unter-Dictionary (oder direkt) an das Ergebnis hängen
                # Direkt flach ins Listing-Dictionary schreiben (wie price):
                res["dist_ubahn_m"] = float(oeffi_res["dist_ubahn_m"][0])
                res["nearest_ubahn"] = str(oeffi_res["nearest_ubahn"][0])
                res["n_tram_lines"] = int(oeffi_res["n_tram_lines"][0])
                res["n_bus_lines"] = int(oeffi_res["n_bus_lines"][0])
                res["n_regionalbus_lines"] = int(oeffi_res["n_regionalbus_lines"][0])
                res["n_sbahn_lines"] = int(oeffi_res["n_sbahn_lines"][0])
                res["n_badnerbahn_lines"] = int(oeffi_res["n_badnerbahn_lines"][0])
                res["n_lines"] = int(oeffi_res["n_lines"][0])
                
            else:
                res["oeffi_features"] = "Keine gültigen Koordinaten gefunden"

            # 5. Ausgeben im gewohnten Stil
            print(f"\n--- BEISPIEL: {source_name.upper()} (Normalisiert inkl. Öffi-Features im RAM) ---")
            print(json.dumps(res, indent=2, ensure_ascii=False))
            break

# --- Anwendung ---
NIGHT_LINE_RE = re.compile(r"^N")
stops = load_stops("../data/external/haltestellen.json")
print_first_valid_with_oeffi("willhaben", db["raw_willhaben"], norm_willhaben, stops)
print_first_valid_with_oeffi("immoscout24", db["raw_immoscout24"], norm_immoscout, stops)

   7213 Haltestellen geladen (Tram 957, Bus 3558, Regionalbus 2459, Ubahn 99, Sbahn 140, Badnerbahn 0)

--- BEISPIEL: WILLHABEN (Normalisiert inkl. Öffi-Features im RAM) ---
{
  "source": "willhaben",
  "id": "2043700366",
  "url": "https://www.willhaben.at/iad/immobilien/d/mietwohnungen/wien/wien-1230-liesing/2-zimmer-mietwohnung-in-perfektastrasse-beste-infrastruktur-2043700366/",
  "title": "2-Zimmer Mietwohnung in Perfektastraße, beste Infrastruktur",
  "plz": 1230,
  "address": "Wien, 23. Bezirk, Liesing",
  "street": null,
  "price": 970.0,
  "area_m2": 50.0,
  "rooms": 2.0,
  "has_outdoor_space": 1,
  "is_rent": 1,
  "is_buy": 0,
  "is_private": 0,
  "is_commercial": 1,
  "lat": 48.144081115722656,
  "lon": 16.291519165039062,
  "dist_ubahn_m": 1813.0,
  "nearest_ubahn": "Perfektastraße",
  "n_tram_lines": 0,
  "n_bus_lines": 6,
  "n_regionalbus_lines": 0,
  "n_sbahn_lines": 12,
  "n_badnerbahn_lines": 0,
  "n_lines": 13
}

--- BEISPIEL: IMMOSCOUT24 (Normalisiert inkl. Öffi-Feat

**Bezirksgrenzen:**

In [10]:
def _first_xy(coords):
    while isinstance(coords[0], (list, tuple)):
        coords = coords[0]
    return coords[0], coords[1]


def _bezirk_number(props):
    for key in ("BEZ", "BEZNR", "BEZ_NR", "BEZIRK", "BEZIRKSNUMMER", "NAMEK_NUM", "BEZ_RZ"):
        for k, v in props.items():
            if k.upper() == key:
                try:
                    nr = int(str(v).strip().lstrip("0") or 0)
                    if 1 <= nr <= 23:
                        return nr
                except ValueError:
                    pass
    for v in props.values():                         # Fallback: "01., Innere Stadt" o. ä.
        m = re.match(r"^\s*0?(\d{1,2})\s*[.,]", str(v))
        if m and 1 <= int(m.group(1)) <= 23:
            return int(m.group(1))
    return None


def load_bezirke(path):
    """Gibt (STRtree, [Polygone], [Bezirksnummern]) zurück oder None, wenn nicht möglich."""
    try:
        from shapely.geometry import shape
        from shapely.ops import transform
        from shapely.strtree import STRtree
        from pyproj import Transformer
    except ImportError:
        print("(Bezirksgrenzen übersprungen: pip install shapely pyproj)")
        return None
    with open(path, encoding="utf-8") as f:
        features = json.load(f)["features"]
    x, y = _first_xy(features[0]["geometry"]["coordinates"])
    projected = abs(x) > 180 or abs(y) > 90
    proj = Transformer.from_crs(BEZIRKE_CRS, "EPSG:4326", always_xy=True).transform if projected else None
    geoms, numbers = [], []
    for ft in features:
        nr = _bezirk_number(ft.get("properties") or {})
        if nr is None:
            print(f"Bezirksnummer nicht erkannt. Properties: {list((features[0].get('properties') or {}).items())[:6]}")
            return None
        g = shape(ft["geometry"])
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", DeprecationWarning)
            geoms.append(transform(proj, g) if proj else g)
        numbers.append(nr)
    print(f"   {len(geoms)} Bezirksgrenzen geladen" + (f" (umgerechnet von {BEZIRKE_CRS})" if projected else ""))
    return STRtree(geoms), geoms, numbers


def bezirk_from_coords(lat, lon, bezirke):
    from shapely.geometry import Point
    tree, geoms, numbers = bezirke
    res = []
    for la, lo in zip(lat, lon):
        if pd.isna(la):
            res.append(pd.NA)
            continue
        hits = tree.query(Point(lo, la), predicate="within")
        res.append(numbers[int(hits[0])] if len(hits) else pd.NA)
    return pd.array(res, dtype="Int64")

Beispiel:

In [11]:
import pandas as pd
BEZIRKE_CRS = "EPSG:31256"

# 1. Bezirksgrenzen laden 
bezirke_tuple = load_bezirke("../data/external/bezirksgrenzen.json")

if bezirke_tuple:
    print("\nBezirke erfolgreich geladen! Teste Zuordnung...")

    # 2. Test-Koordinaten (Beispiel: Rathaus Wien & Stephansdom)
    test_lats = [48.2103, 48.2082]
    test_lons = [16.3566, 16.3738]

    # 3. Funktion testen
    bezirk_ergebnisse = bezirk_from_coords(test_lats, test_lons, bezirke_tuple)

    # 4. Schick ausgeben
    print("\n--- BEISPIEL-AUSGABE: BEZIRKSZUORDNUNG ---")
    for lat, lon, b_nr in zip(test_lats, test_lons, bezirk_ergebnisse):
        print(f"Koordinate ({lat}, {lon}) -> Bezirk: {b_nr}")
else:
    print("Fehler beim Laden der Bezirksgrenzen. Bitte Pfad prüfen.")

   23 Bezirksgrenzen geladen (umgerechnet von EPSG:31256)

Bezirke erfolgreich geladen! Teste Zuordnung...

--- BEISPIEL-AUSGABE: BEZIRKSZUORDNUNG ---
Koordinate (48.2103, 16.3566) -> Bezirk: 1
Koordinate (48.2082, 16.3738) -> Bezirk: 1


## 4. Bereinigung und Speichern in `listings_clean`

Aus der Rohzone entsteht eine Tabelle welche Schritt für Schritt gefiltert, von Duplikaten befreit und um Merkmale ergänzt wird. Das Ergebnis landet in MongoDB in der Collection `listings_clean`

**Zusammenführen der Rohzone zu einer Tabelle mit gemeinsamen Schema:**

In [12]:
def build_records():
    """Liest die Rohzone aus MongoDB und bringt jedes Inserat auf das gemeinsame Schema."""
    stats = {"kein_rent": 0, "projekt": 0}
    records = []
    for source, cfg in SOURCES.items():
        for doc in db[cfg["collection"]].find({}):
            rec = norm_willhaben(doc) if source == "willhaben" else norm_immoscout(doc, stats)
            if rec:
                rec.update(first_seen=doc.get("first_seen"), last_seen=doc.get("last_seen"), status=doc.get("status"))
                records.append(rec)
    print(f"ImmoScout24: {stats['kein_rent']} Nicht-Miet-Inserate und {stats['projekt']} Bauprojekte schon beim Normalisieren aussortiert")
    return records


df = pd.DataFrame(build_records())
print(f"{len(df)} Inserate aus der Rohzone zusammengeführt")
print(df.groupby("source")["id"].count().to_string())

ImmoScout24: 2 Nicht-Miet-Inserate und 6 Bauprojekte schon beim Normalisieren aussortiert
42 Inserate aus der Rohzone zusammengeführt
source
immoscout24    12
willhaben      30


**Filtern nach Duplikaten innerhalb der Quelle, kein Mietinserat, keine Wohnung, ect:**

In [13]:
ROOMS_RANGE = (1, 15)
funnel = [{"Schritt": "Rohzone zusammengeführt", "entfernt": 0, "übrig": len(df)}]


def apply_filter(frame, keep, label):
    out = frame[keep].copy()
    funnel.append({"Schritt": label, "entfernt": len(frame) - len(out), "übrig": len(out)})
    return out


df = apply_filter(df, ~df.duplicated(["source", "id"]), "Duplikate innerhalb der Quelle")
df = apply_filter(df, df["is_rent"] == 1, "kein Mietinserat")
df = apply_filter(df, ~df["title"].fillna("").str.contains(NOT_APARTMENT_RE), "keine Wohnung (Büro, Lokal, Garage ...)")
df = apply_filter(df, df["price"].between(*PRICE_RANGE), "Preis fehlt oder unplausibel")
df = apply_filter(df, df["area_m2"].between(*AREA_RANGE), "Fläche fehlt oder unplausibel")
df = apply_filter(df, df["rooms"].between(*ROOMS_RANGE), "Zimmeranzahl fehlt oder unplausibel")
df = apply_filter(df, df["lat"].between(*VIENNA_LAT) & df["lon"].between(*VIENNA_LON), "Koordinaten fehlen oder liegen außerhalb Wiens")

pd.DataFrame(funnel)

,Schritt,entfernt,übrig
0,Rohzone zusammengeführt,0,42
1,Duplikate innerhalb der Quelle,0,42
2,kein Mietinserat,0,42
3,"keine Wohnung (Büro, Lokal, Garage ...)",2,40
4,Preis fehlt oder unplausibel,0,40
5,Fläche fehlt oder unplausibel,0,40
6,Zimmeranzahl fehlt oder unplausibel,0,40
7,Koordinaten fehlen oder liegen außerhalb Wiens,2,38


**Filtern nach Duplikaten zwischen den Portalen:**

Dieselbe Wohnung steht oft auf Willhaben und ImmoScout24. Zwei Inserate gelten als dieselbe Wohnung, wenn
- Zimmerzahl gleich ist und die Fläche um höchstens 1 m² abweicht (immer), **und**
- entweder **streng**: Preis ±3 % und Koordinaten höchstens 75 m auseinander,
- oder **über den Titel**: Der Titel ist ähnlich (Ähnlichkeit ab 0,8) und es genügen Preis ±5 % und höchstens 150 m Abstand.

Verglichen wird nur **zwischen** den Portalen, nicht innerhalb eines Portals. So werden baugleiche Wohnungen im selben Haus nicht fälschlich zusammengefasst. Behalten wird das Willhaben-Inserat, `n_portals` hält fest, auf wie vielen Portalen die Wohnung steht.

In [14]:
from difflib import SequenceMatcher

DUP = dict(area_tol=1.0, strict_price=0.03, strict_dist=75, relaxed_price=0.05, relaxed_dist=150, title_sim=0.8)


def normalize_title(text):
    text = re.sub(r"[^\w\s]", " ", str(text or "").lower())
    return re.sub(r"\s+", " ", text).strip()


def haversine_m(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6_371_000 * np.arcsin(np.sqrt(a))


def find_cross_portal_duplicates(frame):
    wh = frame[frame["source"] == "willhaben"]
    im = frame[frame["source"] == "immoscout24"]
    wh_by_rooms = {r: g for r, g in wh.groupby("rooms")}
    pairs = []
    for i, row in im.iterrows():
        cand = wh_by_rooms.get(row["rooms"])
        if cand is None:
            continue
        area_ok = (cand["area_m2"] - row["area_m2"]).abs() <= DUP["area_tol"]
        price_diff = (cand["price"] - row["price"]).abs() / row["price"]
        dist = pd.Series(haversine_m(row["lat"], row["lon"], cand["lat"].values, cand["lon"].values), index=cand.index)
        relaxed = area_ok & (price_diff <= DUP["relaxed_price"]) & (dist <= DUP["relaxed_dist"])
        for j in cand.index[relaxed]:
            strict = price_diff[j] <= DUP["strict_price"] and dist[j] <= DUP["strict_dist"]
            sim = SequenceMatcher(None, normalize_title(row["title"]), normalize_title(cand.at[j, "title"])).ratio()
            if strict or sim >= DUP["title_sim"]:
                pairs.append(dict(idx_wh=j, idx_im=i, dist_m=round(dist[j]), price_diff_pct=round(100 * price_diff[j], 1),
                                  area_diff=round(abs(cand.at[j, "area_m2"] - row["area_m2"]), 1),
                                  title_sim=round(sim, 2), matched_by="streng" if strict else "titel"))
    return pd.DataFrame(pairs)


candidates = find_cross_portal_duplicates(df)
print(f"{len(candidates)} Kandidatenpaare gefunden")
if len(candidates):
    print(candidates["matched_by"].value_counts().to_string())

    # Jedes Inserat darf nur zu einem Paar gehören: die besten Paare zuerst (ähnlichster Titel, kleinster Abstand)
    used_wh, used_im, chosen = set(), set(), []
    for _, p in candidates.sort_values(["title_sim", "dist_m"], ascending=[False, True]).iterrows():
        if p["idx_wh"] in used_wh or p["idx_im"] in used_im:
            continue
        used_wh.add(p["idx_wh"]); used_im.add(p["idx_im"]); chosen.append(p)
    matches = pd.DataFrame(chosen)
    print(f"{len(matches)} Duplikate zwischen Willhaben und ImmoScout24 (1:1 zugeordnet)")

    # Stichprobe zur Kontrolle
    sample = matches.head(8).copy()
    sample["titel_willhaben"] = df.loc[sample["idx_wh"], "title"].str[:45].values
    sample["titel_immoscout"] = df.loc[sample["idx_im"], "title"].str[:45].values
    display(sample[["matched_by", "dist_m", "price_diff_pct", "area_diff", "title_sim", "titel_willhaben", "titel_immoscout"]])

    df["n_portals"] = 1
    df.loc[matches["idx_wh"].astype(int), "n_portals"] = 2
    df = apply_filter(df, ~df.index.isin(matches["idx_im"].astype(int)), "Duplikate zwischen den Portalen")
else:
    df["n_portals"] = 1

0 Kandidatenpaare gefunden


**Merkmale ergänzen:**

- **Bezirk:** aus den Koordinaten über die Bezirksgrenzen (genauer als die PLZ, weil PLZ-Gebiete nicht immer mit den Bezirksgrenzen übereinstimmen). Nur wenn keine Zuordnung möglich ist, wird der Bezirk aus der PLZ abgeleitet.
- **Preis pro m²** und der Vergleich mit dem **Median des Bezirks** (nur aktive Inserate, ab 5 Inseraten pro Bezirk).
- **Öffi-Merkmale:** Distanz zur nächsten U-Bahn, Anzahl der Linien je Verkehrsmittel im Umkreis (`OEFFI_RADIUS_M`) und ein Flag für „U-Bahn innerhalb von 500 m".

In [15]:
UBAHN_NEAR_M = 500


def bezirk_from_plz(plz):
    if pd.isna(plz):
        return pd.NA
    plz = int(plz)
    return plz // 10 % 100 if 1010 <= plz <= 1230 and plz % 10 == 0 else pd.NA


df["plz"] = df["plz"].astype("Int64")

# Bezirk: primär aus den Koordinaten, sonst aus der PLZ
bezirke = load_bezirke(BEZIRKE_FILE)
by_plz = df["plz"].map(bezirk_from_plz).astype("Int64")
if bezirke:
    by_geo = pd.Series(bezirk_from_coords(df["lat"].tolist(), df["lon"].tolist(), bezirke), index=df.index, dtype="Int64")
    both = by_plz.notna() & by_geo.notna()
    print(f"Bezirk aus Koordinaten und PLZ vergleichbar bei {int(both.sum())} Inseraten, abweichend bei {int((by_plz[both] != by_geo[both]).sum())}")
    df["bezirk"] = by_geo.fillna(by_plz)
else:
    df["bezirk"] = by_plz
print(f"Ohne Bezirk: {int(df['bezirk'].isna().sum())} Inserate")

# Preis pro m² und Vergleich mit dem Bezirksmedian
df["price_per_m2"] = (df["price"] / df["area_m2"]).round(2)
active_ppm2 = df["price_per_m2"].where(df["status"] == "active")
grp = active_ppm2.groupby(df["bezirk"])
enough = grp.transform("count") >= MIN_LISTINGS_PER_BEZIRK
df["bezirk_median_ppm2"] = grp.transform("median").where(enough).round(2)
df["ppm2_vs_bezirk"] = (df["price_per_m2"] / df["bezirk_median_ppm2"]).round(3)

# Öffi-Merkmale
NIGHT_LINE_RE = re.compile(r"^N")        # Nachtlinien ignorieren (wie in der Beispielzelle oben)
stops = load_stops(STOPS_FILE)
feats = oeffi_features(df["lat"].values, df["lon"].values, stops)
for col, values in feats.items():
    df[col] = values
df["ubahn_within_500m"] = (df["dist_ubahn_m"] <= UBAHN_NEAR_M).astype(int)

FINAL_COLS = ["source", "id", "url", "title", "plz", "bezirk", "address",
              "price", "area_m2", "rooms", "price_per_m2", "bezirk_median_ppm2", "ppm2_vs_bezirk",
              "has_outdoor_space", "is_private", "is_commercial", "n_portals",
              "lat", "lon", "dist_ubahn_m", "nearest_ubahn", "ubahn_within_500m",
              "n_ubahn_lines", "n_tram_lines", "n_bus_lines", "n_regionalbus_lines",
              "n_sbahn_lines", "n_badnerbahn_lines", "n_lines",
              "status", "first_seen", "last_seen"]
df = df[FINAL_COLS]
print(f"\n{len(df)} bereinigte Inserate, {len(FINAL_COLS)} Spalten")
df.head()

   23 Bezirksgrenzen geladen (umgerechnet von EPSG:31256)
Bezirk aus Koordinaten und PLZ vergleichbar bei 38 Inseraten, abweichend bei 0
Ohne Bezirk: 0 Inserate
   7213 Haltestellen geladen (Tram 957, Bus 3558, Regionalbus 2459, Ubahn 99, Sbahn 140, Badnerbahn 0)

38 bereinigte Inserate, 32 Spalten


,source,id,url,title,plz,bezirk,address,price,area_m2,rooms,...,n_ubahn_lines,n_tram_lines,n_bus_lines,n_regionalbus_lines,n_sbahn_lines,n_badnerbahn_lines,n_lines,status,first_seen,last_seen
0,willhaben,2043700366,https://www.willhaben.at/iad/immobilien/d/miet...,"2-Zimmer Mietwohnung in Perfektastraße, beste ...",1230,23,"Wien, 23. Bezirk, Liesing",970.00,50.0,2.0,...,0,0,6,0,12,0,13,active,2026-11-28 08:27:00.070,2026-11-28 08:51:57.434
1,willhaben,1479216183,https://www.willhaben.at/iad/immobilien/d/miet...,Ihr exklusiver Wohnsitz in Wien - Prestige & K...,1030,3,"Wien, 03. Bezirk, Landstraße",5988.90,244.0,5.0,...,0,4,4,0,13,0,14,active,2026-11-28 08:27:00.070,2026-11-28 08:51:57.434
2,willhaben,955774944,https://www.willhaben.at/iad/immobilien/d/miet...,vollmöbliert im Herzen Margaretens - ab Jänner...,1050,5,"Wien, 05. Bezirk, Margareten",774.35,59.0,2.0,...,0,0,3,0,0,0,3,active,2026-11-28 08:27:00.070,2026-11-28 08:51:57.434
3,willhaben,1409788638,https://www.willhaben.at/iad/immobilien/d/miet...,2 Zimmer Wohnung-Wiener Wohnen,1210,21,"Wien, 21. Bezirk, Floridsdorf",508.68,48.0,2.0,...,0,2,7,1,0,0,8,active,2026-11-28 08:27:00.070,2026-11-28 08:51:57.434
4,willhaben,886221532,https://www.willhaben.at/iad/immobilien/d/miet...,Ihre neue Wohnung wartet schon - moderner Erst...,1030,3,"Wien, 03. Bezirk, Landstraße",1799.00,74.0,3.0,...,0,5,8,0,7,0,9,active,2026-11-28 08:27:00.070,2026-11-28 08:51:57.434


# 5. Bereinigte Zone in MongoDB speichern

Die Collection `listings_clean` ist abgeleitet und wird bei jedem Lauf komplett neu aufgebaut. Der Schlüssel `_id` ist `quelle:id`.

In [16]:
def to_documents(frame):
    """DataFrame -> Liste von Dokumenten (fehlende Werte werden zu None, damit MongoDB sie speichern kann)."""
    cleaned = frame.astype(object).where(frame.notna(), None)
    docs = cleaned.to_dict("records")
    for d in docs:
        d["_id"] = f"{d['source']}:{d['id']}"
    return docs


clean_coll = db["listings_clean"]
clean_coll.delete_many({})
clean_coll.insert_many(to_documents(df))
clean_coll.create_index([("bezirk", 1), ("status", 1)])
clean_coll.create_index([("source", 1)])
print(f"{clean_coll.count_documents({})} Dokumente in listings_clean gespeichert")

38 Dokumente in listings_clean gespeichert


# 6. Kontrolle

**Wie viele Inserate bleiben nach jedem Schritt übrig?**

In [17]:
pd.DataFrame(funnel)

,Schritt,entfernt,übrig
0,Rohzone zusammengeführt,0,42
1,Duplikate innerhalb der Quelle,0,42
2,kein Mietinserat,0,42
3,"keine Wohnung (Büro, Lokal, Garage ...)",2,40
4,Preis fehlt oder unplausibel,0,40
5,Fläche fehlt oder unplausibel,0,40
6,Zimmeranzahl fehlt oder unplausibel,0,40
7,Koordinaten fehlen oder liegen außerhalb Wiens,2,38


**Überblick je Quelle** (nur aktive Inserate):

In [18]:
active = df[df["status"] == "active"]
active.groupby("source").agg(
    anzahl=("id", "count"),
    median_preis=("price", "median"),
    median_preis_m2=("price_per_m2", "median"),
    median_flaeche=("area_m2", "median"),
    anteil_freiflaeche=("has_outdoor_space", "mean"),
    anteil_privat=("is_private", "mean"),
    median_dist_ubahn_m=("dist_ubahn_m", "median"),
).round(2)

,anzahl,median_preis,median_preis_m2,median_flaeche,anteil_freiflaeche,anteil_privat,median_dist_ubahn_m
source,,,,,,,
immoscout24,9,2490.00,23.94,66.0,0.89,0.00,582.0
willhaben,29,1284.99,23.79,54.0,0.83,0.24,611.0


**Direkt in MongoDB abfragen** (Anzahl und durchschnittlicher Preis pro m² je Bezirk, nur aktive Inserate):

In [19]:
pipeline = [
    {"$match": {"status": "active"}},
    {"$group": {"_id": "$bezirk", "anzahl": {"$sum": 1}, "avg_preis_m2": {"$avg": "$price_per_m2"}}},
    {"$sort": {"_id": 1}},
]
result = pd.DataFrame(list(clean_coll.aggregate(pipeline))).rename(columns={"_id": "bezirk"})
result["avg_preis_m2"] = result["avg_preis_m2"].round(2)
result

,bezirk,anzahl,avg_preis_m2
0,2,2,26.64
1,3,10,24.19
2,4,1,23.94
3,5,2,16.93
4,9,1,32.99
5,10,2,25.16
6,12,2,24.46
7,13,1,25.17
8,14,1,21.45
9,15,1,25.43


**Fehlende Werte** in den wichtigsten Spalten (Anteil in %):

In [20]:
key_cols = ["price", "area_m2", "rooms", "price_per_m2", "bezirk", "ppm2_vs_bezirk", "dist_ubahn_m", "n_lines"]
(df[key_cols].isna().mean() * 100).round(1).to_frame("fehlend %")

,fehlend %
price,0.0
area_m2,0.0
rooms,0.0
price_per_m2,0.0
bezirk,0.0
ppm2_vs_bezirk,73.7
dist_ubahn_m,0.0
n_lines,0.0
